Gold reproducibility notebook

- Configure `DATA_DOI`, `MODEL_DOI`, filenames, and switches in the next cell.
- Downloads artifacts from Zenodo if missing, verifies hashes, sets deterministic seeds, trains or loads model, captures environment, and writes a provenance manifest.
- Assumes dataset and model are archived in Zenodo (Bronze met).


In [ ]:
# Configuration and reproducibility helpers
from __future__ import annotations
import os
import sys
import json
import hashlib
import random
import pathlib
from typing import Optional

import numpy as np

# --- Config ---
DATA_DOI = os.environ.get("DATA_DOI", "10.5281/zenodo.xxxxx")  # replace
MODEL_DOI = os.environ.get("MODEL_DOI", "")                     # optional
DATA_FILENAME = os.environ.get("DATA_FILENAME", "simple_dataset.csv")
MODEL_FILENAME = os.environ.get("MODEL_FILENAME", "linear_regression.pkl")
ARTIFACTS_DIR = os.environ.get("ARTIFACTS_DIR", "artifacts")
DATA_DIR = os.environ.get("DATA_DIR", "Data")
VERBOSE = True
USE_DOWNLOADED_MODEL = os.environ.get("USE_DOWNLOADED_MODEL", "0") == "1"

# Optional integrity checks
EXPECTED_DATA_SHA256 = os.environ.get("EXPECTED_DATA_SHA256", "")
EXPECTED_MODEL_SHA256 = os.environ.get("EXPECTED_MODEL_SHA256", "")

# --- Reproducibility seeds ---
GLOBAL_SEED = int(os.environ.get("GLOBAL_SEED", "674"))
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

# --- Utils ---
def ensure_dir(path: str) -> None:
    pathlib.Path(path).mkdir(parents=True, exist_ok=True)


def sha256_of_file(path: str) -> str:
    sha = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            sha.update(chunk)
    return sha.hexdigest()


def verify_hash(path: str, expected_sha256: str) -> bool:
    if not expected_sha256:
        return True
    actual = sha256_of_file(path)
    if VERBOSE:
        print(f"SHA256 for {path}: {actual}")
    return actual.lower() == expected_sha256.lower()


def zenodo_download(doi: str, filename: str, dest_dir: str) -> str:
    import urllib.request
    ensure_dir(dest_dir)
    record_id = doi.split(".")[-1].replace("zenodo/", "").replace("zenodo-", "").replace("zenodo", "").replace("/", "")
    url = f"https://zenodo.org/records/{record_id}/files/{filename}?download=1"
    dest_path = os.path.join(dest_dir, filename)
    if VERBOSE:
        print(f"Downloading {url} -> {dest_path}")
    urllib.request.urlretrieve(url, dest_path)
    return dest_path

ensure_dir(ARTIFACTS_DIR)
ensure_dir(DATA_DIR)
print("Configuration loaded. Seeds set.")


In [ ]:
import os
import pandas as pd

# Load dataset from local Data/ or download from Zenodo
local_path = os.path.join(DATA_DIR, DATA_FILENAME)
if not os.path.exists(local_path):
    if not DATA_DOI or DATA_DOI.endswith("xxxxx"):
        raise RuntimeError("DATA_DOI not set to a real Zenodo DOI and local data not found.")
    local_path = zenodo_download(DATA_DOI, DATA_FILENAME, DATA_DIR)
    if EXPECTED_DATA_SHA256 and not verify_hash(local_path, EXPECTED_DATA_SHA256):
        raise RuntimeError("Downloaded data hash mismatch; refusing to proceed.")

print(f"Using dataset at: {local_path}")
df = pd.read_csv(local_path)
print(df.head())

Generated DataFrame Head (with correlated Score):
   UserID  Age      City  Salary  Score
0       1   56   Chicago   91717  55.16
1       2   46   Chicago   95859  59.17
2       3   32  New York   71309  51.28
3       4   60   Chicago  108734  71.19
4       5   25   Chicago  115467  54.51

Successfully saved dataset with correlated score to 'Data/simple_dataset.csv'


In [ ]:
# Preview the loaded dataframe
print(df.head())

   UserID  Age      City  Salary  Score
0       1   56   Chicago   91717  55.16
1       2   46   Chicago   95859  59.17
2       3   32  New York   71309  51.28
3       4   60   Chicago  108734  71.19
4       5   25   Chicago  115467  54.51


In [ ]:
import os
import pickle
import time
import subprocess
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

start_time = time.time()

# Preprocess
df_processed = pd.get_dummies(df, columns=['City'], drop_first=True)
X = df_processed.drop(['UserID', 'Score'], axis=1)
y = df_processed['Score']

# Deterministic split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=GLOBAL_SEED
)

model_path = os.path.join(ARTIFACTS_DIR, MODEL_FILENAME)

# Optionally download an existing model
if USE_DOWNLOADED_MODEL:
    if not MODEL_DOI:
        raise RuntimeError("USE_DOWNLOADED_MODEL=1 but MODEL_DOI is empty.")
    model_path = zenodo_download(MODEL_DOI, MODEL_FILENAME, ARTIFACTS_DIR)
    if EXPECTED_MODEL_SHA256 and not verify_hash(model_path, EXPECTED_MODEL_SHA256):
        raise RuntimeError("Downloaded model hash mismatch; refusing to proceed.")
    with open(model_path, "rb") as f:
        model = pickle.load(f)
else:
    # Train deterministically
    model = LinearRegression()
    model.fit(X_train, y_train)
    # Save model
    with open(model_path, "wb") as f:
        pickle.dump(model, f)
    # Report model hash
    model_sha = sha256_of_file(model_path)
    print(f"Saved model to {model_path}")
    print(f"Model SHA256: {model_sha}")

# Evaluate
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("--- Model Performance ---")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R²): {r2:.2f}")

print("\n--- Model Coefficients ---")
coeffs = pd.DataFrame(model.coef_, X_train.columns, columns=['Coefficient'])
print(coeffs)

# Write environment freeze
ensure_dir(ARTIFACTS_DIR)
req_path = os.path.join(ARTIFACTS_DIR, "requirements-gold.txt")
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
with open(req_path, "w") as f:
    f.write(freeze)
print(f"Saved environment to {req_path}")

# Git metadata (best-effort)
try:
    import subprocess as sp
    commit = sp.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    branch = sp.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip()
except Exception:
    commit = ""
    branch = ""

# Manifest with provenance
manifest = {
    "data": {
        "path": os.path.join(DATA_DIR, DATA_FILENAME),
        "sha256": sha256_of_file(os.path.join(DATA_DIR, DATA_FILENAME)) if os.path.exists(os.path.join(DATA_DIR, DATA_FILENAME)) else "",
        "doi": DATA_DOI,
    },
    "model": {
        "path": model_path,
        "sha256": sha256_of_file(model_path) if os.path.exists(model_path) else "",
        "doi": MODEL_DOI,
    },
    "metrics": {"mse": float(mse), "r2": float(r2)},
    "env": {
        "requirements": req_path,
        "python": sys.version,
        "numpy": np.__version__,
    },
    "git": {"commit": commit, "branch": branch},
    "runtime_seconds": round(time.time() - start_time, 3),
}

man_path = os.path.join(ARTIFACTS_DIR, "manifest.json")
with open(man_path, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"Saved manifest to {man_path}")


Verification and next steps

- Confirm `manifest.json` lists the expected data/model `sha256` and DOIs.
- Re-run with `USE_DOWNLOADED_MODEL=1` to verify artifact loading path.
- Upload `manifest.json`, `requirements-gold.txt`, and model artifact to Zenodo to complete the Gold package.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

df_processed = pd.get_dummies(df, columns=['City'], drop_first=True)

X = df_processed.drop(['UserID', 'Score'], axis=1)
y = df_processed['Score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=674)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)


mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("--- Model Performance ---")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R²): {r2:.2f}")


print("\n--- Model Coefficients ---")

coeffs = pd.DataFrame(model.coef_, X_train.columns, columns=['Coefficient'])
print(coeffs)


--- Model Performance ---
Mean Squared Error (MSE): 29.99
R-squared (R²): 0.75

--- Model Coefficients ---
                  Coefficient
Age                  0.564729
Salary               0.000333
City_Houston        -2.968734
City_Los Angeles     8.420214
City_New York       12.329229
City_Phoenix        -8.096262
